# Kirchhoff-Love modes of a clamped hexagonal membrane

We will calculate the vibrational modes of a flat, free hexagonal membrane with a spatially varying bending stiffness, clamped to an external surface at three points. The modes are calculated using Kirchhoff-Love plate theory.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from hcipy import *


The Kirchhoff-Love theory describes the deflection of a thin elastic plate. For free vibration, the deflection $u(x)$ is the solution of the generalized eigenvalue problem

$$ K u = \omega^2 M u, $$

where $K$ is the bending-stiffness matrix and $M$ the mass matrix. Both are assembled on the grid from the bending stiffness $D(x)$ of the membrane. The resulting eigenvectors are the mode shapes, sorted by ascending natural frequency $\omega$. In HCIPy this is done by `make_kirchhoff_love_basis()`, which takes the bending stiffness as a `Field` on a regular Cartesian grid.

We start with a hexagonal membrane, defined through `make_hexagonal_aperture()`. The grid has to extend beyond the membrane; the membrane itself is left unclamped at its edge, and the free (natural) edge conditions emerge from the variational discretization of the bending energy.


In [ ]:
circum_diameter = 1

grid = make_pupil_grid(128, 1.2)
bending_stiffness = make_hexagonal_aperture(circum_diameter)(grid)

imshow_field(bending_stiffness, cmap='Greys')
plt.title('Bending stiffness of the hexagonal membrane')
plt.show()


The membrane is clamped to an external surface at three points. These clamping points are circular patches, centered about half the circumdiameter of the hexagon, in the direction of three alternating vertices. At these points the deflection is required to be zero, which is passed to `make_kirchhoff_love_basis()` through the `fixed_points` keyword.


In [ ]:
patch_diameter = 0.1
vertex_angles = np.deg2rad([90, 210, 330])

fixed_points = Field(np.zeros(grid.size), grid)

for angle in vertex_angles:
    center = circum_diameter / 4 * np.array([np.cos(angle), np.sin(angle)])
    patch = make_circular_aperture(patch_diameter, center=center)(grid)
    fixed_points += patch > 0.5

fixed_points = fixed_points > 0

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
imshow_field(bending_stiffness, cmap='Greys')
plt.title('Membrane')

plt.subplot(1, 2, 2)
imshow_field(fixed_points, cmap='Greys')
plt.title('Clamping points')

plt.show()


Now we can calculate the modes of the membrane.


In [ ]:
num_modes = 9

modes, frequencies = make_kirchhoff_love_basis(bending_stiffness, num_modes, fixed_points=fixed_points, return_frequencies=True)

for i, f in enumerate(frequencies):
    print('Mode {}: frequency = {:.3e} Hz'.format(i, f))


In [ ]:
plt.figure(figsize=(14, 12))

for i, mode in enumerate(modes):
    m = mode / np.max(np.abs(mode))

    plt.subplot(3, 3, i + 1)
    imshow_field(m, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.title('Mode {}'.format(i))

plt.tight_layout()
plt.show()


Finally, we animate the modes. Each mode oscillates at its natural frequency. If the bending stiffness is in N m, the mass density in kg / m^2, and the grid coordinates in meters, the frequencies are the physical resonance frequencies in Hz. Here all quantities are dimensionless, so we animate the modes with the ratio of the frequencies, so that every mode completes a whole number of oscillation periods per animation.

We use the `FFMpegWriter` class to write the animation to a video file. The resulting animation is shown inline in the notebook by evaluating the writer object.


In [ ]:
N = int(grid.regular_coords[1][0])

aperture_mask = np.asarray(bending_stiffness).reshape((N, N)) > 0.5
omega = frequencies / frequencies[0]

plt.figure(figsize=(12, 10))
anim = FFMpegWriter('kirchhoff_love_modes.mp4', framerate=10)

num_frames = 30
for i in range(num_frames):
    t = 2 * np.pi * i / num_frames

    plt.clf()
    plt.suptitle('Kirchhoff-Love modes of the clamped hexagonal membrane')

    for j, mode in enumerate(modes):
        m = np.asarray(mode).reshape((N, N)) / np.max(np.abs(mode))
        m = np.where(aperture_mask, m * np.cos(omega[j] * t), np.nan)

        plt.subplot(3, 3, j + 1)
        plt.imshow(m, cmap='RdBu_r', vmin=-1, vmax=1)
        plt.title('Mode {}'.format(j))
        plt.axis('off')

    plt.tight_layout()
    anim.add_frame()

plt.close()
anim.close()

# Show the animation
anim


In [ ]:
# Remove created files.
import os
os.remove('kirchhoff_love_modes.mp4')